In [1]:
import time
import pickle

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# from sklearn.linear_model import SGDRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

from dice.agents import Smart

In [2]:
# Load the trained agent
print("Loading the trained Agent.")
fp = "./dice/trained_agents/trained_agent_1709936115_25000.pkl"
with open(fp, "rb") as fin:
    agent: Smart = pickle.load(fin)

Loading the trained Agent.


In [ ]:
# Format the data for the model
print("Encoding the training data.")
encoder = OneHotEncoder(sparse_output=True)
rows: list[tuple[str, ...]] = []
for state, action in agent.X:
    rows.append(tuple(state.split("+")) + (action,))

In [33]:
NUM_ROWS_TO_USE = len(rows)
features = pd.DataFrame(rows, columns=["card", "turn_score", "roll", "hold", "action"]).sample(NUM_ROWS_TO_USE)

In [34]:
# Encode the categorical features
numerical_columns = ["turn_score"]
numerical_features = features[numerical_columns].assign(
    **{
        nc: lambda df: df[nc].str.split("_").str.get(-1).astype(int)
        for nc in numerical_columns
    }
)
features = features.drop(columns=numerical_columns)
encoded_categorical_features = encoder.fit_transform(features)

In [35]:
# Recombine the numerical and categorical features
X = hstack((numerical_features, encoded_categorical_features))
y = pd.Series(agent.y).iloc[features.index]

# Split the data for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=31
)

In [36]:
# Train a model to estimate the reward for taking a particular action from a given state
print("Training the estimator.")
estimator = RandomForestRegressor(verbose=1, n_jobs=-1)
# estimator = SGDRegressor(verbose=1)
# estimator = SVR(verbose=True)
estimator.fit(X_train, y_train)
estimator.verbose = 0

Training the estimator.


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  6.9min finished


In [37]:
# Evaluate the model
y_pred = estimator.predict(X_test)

print(f"{mean_squared_error(y_test, y_pred)=:0.3f}")
print(f"{r2_score(y_test, y_pred)=:0.3f}")

mean_squared_error(y_test, y_pred)=1.850
r2_score(y_test, y_pred)=0.632


In [39]:
# Save the trained encoder and model
print("Saving the trained estimator.")
with open(
    f"./dice/dice_trained_encoder_{estimator.__repr__(N_CHAR_MAX=25)}_{int(time.time())}.pkl",
    "wb",
) as fout:
    pickle.dump(encoder, fout)
with open(
    f"./dice/dice_trained_estimator_{estimator.__repr__(N_CHAR_MAX=25)}_{int(time.time())}.pkl",
    "wb",
) as fout:
    pickle.dump(estimator, fout)

print("Done!")

Saving the trained estimator.
Done!


In [43]:
!rm dice/dice_trained_estimator_RandomForest...r\(n_jobs\=-1\)_1710022280.pkl

In [42]:
!ls -l dice/

total 2490472
-rw-r--r--   1 john  staff      40159 Dec 13 23:20 Figure_1.png
-rw-r--r--   1 john  staff          0 Nov 28 23:29 __init__.py
drwxr-xr-x  15 john  staff        480 Mar  9 07:48 __pycache__
-rw-r--r--   1 john  staff      15582 Mar  9 07:48 agents.py
-rw-r--r--   1 john  staff       2308 Mar  2 23:18 cards.py
-rw-r--r--   1 john  staff      28349 Mar  8 23:29 dice_trained_encoder.pkl
-rw-r--r--   1 john  staff      28349 Mar  9 14:11 dice_trained_encoder_RandomForest...r(n_jobs=-1)_1710022280.pkl
-rw-r--r--   1 john  staff      28349 Mar  9 14:13 dice_trained_encoder_RandomForest...r(n_jobs=-1)_1710022423.pkl
-rw-r--r--   1 john  staff       8203 Mar  9 07:49 dice_trained_encoder_SVR(verbose=True).pkl
-rw-------   1 john  staff  423937057 Mar  8 22:38 dice_trained_estimator.pkl
-rw-r--r--   1 john  staff  424291585 Mar  9 14:11 dice_trained_estimator_RandomForest...r(n_jobs=-1)_1710022280.pkl
-rw-r--r--   1 john  staff  424291585 Mar  9 14:13 dice_trained_estimator_Random